# Session 5: seasons and a loan
**A. Real-world problem.** The made-up bakery of Sessions 2 to 4 sells the same number of loaves
every month and pays for everything out of its own pocket. Real bakeries sell far more in
December than in February, and most borrow part of the start-up money and repay it in equal
monthly instalments. Over a whole year a seasonal bakery can earn exactly as much as a flat
one and still run out of cash in March, because the instalment is due every month but the
customers are not. Today you build a 12-month demand pattern and a loan, follow the running
cash total month by month, and ask how much extra cash the owner needs.

*General education only: the bakery, the loan and their numbers are made up, and this is not
financial advice.*

In [ ]:
import math
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

F, P, V, Q = 1200.0, 3.50, 1.40, 800.0          # as in Sessions 2 to 4: planned profit 480 dollars a month
STARTUP = 6000.0                                # start-up cost of Session 3's payback example
MONTH_NAMES = ("Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec")
# made-up seasonal factors (the average is exactly 1.0): quiet in winter, a December peak
SEASON = (0.85, 0.80, 0.90, 0.95, 1.00, 0.95, 0.90, 0.90, 1.00, 1.05, 1.15, 1.55)

## Step 1: a 12-month demand pattern
A **seasonal factor** says how a month's demand compares with the yearly average: 1.55 means
55 % more than average. We require the 12 factors to average exactly 1.0, so that the pattern
moves sales *around* the yearly average and does not add or remove any. Then each month's
profit is (p - v) x (planned demand x factor) - F, and because that is a straight line in the
factor, the twelve profits add up to exactly twelve times the flat monthly profit.

In [ ]:
def check_season(season):
    if len(season) != 12:
        raise ValueError("a seasonal pattern needs 12 factors")
    if min(season) < 0:
        raise ValueError("a seasonal factor cannot be negative")
    if abs(sum(season) / 12.0 - 1.0) > 1e-9:
        raise ValueError("the 12 seasonal factors must average 1.0")
    return tuple(season)


def scale_season(season, k):
    """Stronger (k > 1) or weaker (k < 1) seasons; k = 0 is a flat year. The average stays 1.0."""
    new = [1.0 + k * (x - 1.0) for x in check_season(season)]
    if min(new) < 0:
        raise ValueError("that k would make a seasonal factor negative")
    return check_season(new)


def monthly_profits(season=SEASON, months=24, start=1, p=P, v=V, d_mu=Q, fixed=F):
    """Profit of each month for `months` months; the pattern repeats every year; `start` is the opening month (1 = January)."""
    season = check_season(season)
    if not 1 <= start <= 12:
        raise ValueError("the opening month must be 1 to 12")
    return np.array([(p - v) * d_mu * season[(start - 1 + m) % 12] - fixed for m in range(months)])

## Step 2: a loan repaid in equal monthly instalments
Borrow L at an annual rate `apr` (a nominal rate, so the monthly rate is r = apr / 12) and repay
it in n equal instalments at the end of each month. The instalment that clears the debt exactly is
`L r / (1 - (1 + r)^-n)`, and with no interest it is simply L / n. Each instalment pays that
month's interest (balance x r) first and the rest reduces the balance. (The openloan repo
explains this in detail.)

In [ ]:
def loan_payment(principal, apr, n):
    if n < 1 or principal < 0 or apr < 0:
        raise ValueError("need a positive term and a non-negative principal and rate")
    r = apr / 12.0
    return principal / n if r == 0 else principal * r / (1.0 - (1.0 + r) ** -n)


def loan_schedule(principal, apr, n):
    """One row per month: instalment, interest, repaid part and balance left."""
    pay, r, bal, rows = loan_payment(principal, apr, n), apr / 12.0, principal, []
    for m in range(1, n + 1):
        interest = bal * r
        repaid = pay - interest
        bal -= repaid
        rows.append({"month": m, "payment": pay, "interest": interest, "repaid": repaid, "balance": 0.0 if abs(bal) < 1e-9 else bal})
    return rows


def total_interest(principal, apr, n):
    return loan_payment(principal, apr, n) * n - principal

## Step 3: the running cash total, month by month
At month 0 the owner pays the part of the start-up cost that is *not* borrowed: `startup - loan`
(this is the first running total, minus that amount). Each month the cash changes by that
month's profit minus the loan instalment (while the loan lasts). The **extra cash needed** is
how far the running total falls *below its starting point*: the owner must then add money from
somewhere else. It is zero when the cash never falls below where it began.

In [ ]:
def running_total(profits, startup=STARTUP, loan=0.0, apr=0.09, term=12):
    """Cash after each month (the owner's equity startup - loan is already spent at month 0)."""
    profits = np.asarray(profits, dtype=float)
    pay = loan_payment(loan, apr, term) if loan > 0 else 0.0
    inst = np.where(np.arange(len(profits)) < term, pay, 0.0)
    return -(startup - loan) + np.cumsum(profits - inst)


def extra_cash_needed(profits, loan=0.0, apr=0.09, term=12):
    """Largest fall of the running total below its starting level (dollars, 0 if it never falls)."""
    pay = loan_payment(loan, apr, term) if loan > 0 else 0.0
    inst = np.where(np.arange(len(profits)) < term, pay, 0.0)
    return float(max(0.0, -np.min(np.cumsum(np.asarray(profits, dtype=float) - inst))))


def lowest_month(profits, loan=0.0, apr=0.09, term=12):
    """(month number, change of the running total since the start) at the lowest point."""
    pay = loan_payment(loan, apr, term) if loan > 0 else 0.0
    inst = np.where(np.arange(len(profits)) < term, pay, 0.0)
    cs = np.cumsum(np.asarray(profits, dtype=float) - inst)
    return int(np.argmin(cs)) + 1, float(np.min(cs))


def equity_recovered_month(profits, startup=STARTUP, loan=0.0, apr=0.09, term=12):
    """First month in which the running total is back to zero (None if not within the profits given)."""
    ok = np.nonzero(running_total(profits, startup, loan, apr, term) >= -1e-9)[0]
    return int(ok[0]) + 1 if len(ok) else None

## Step 4: the start month matters
The pattern repeats every year, so *when* the bakery opens decides whether the weak months
come before or after the strong ones. Opening in January puts the worst months (Jan to Mar)
first, with no December cash in hand; opening in April puts December before the second winter.

In [ ]:
def extra_cash_by_start(loan, apr, term, season=SEASON, months=24):
    return {s: extra_cash_needed(monthly_profits(season, months, s), loan, apr, term) for s in range(1, 13)}

## Step 5: when demand is also random
Add the randomness of Session 3: in each month the demand is the seasonal mean plus a normal
error (sd 150 loaves, months independent). We draw many years (seeded) and report the share of
years in which the owner needs more than a given buffer of extra cash.

In [ ]:
def simulate_cash(n=5000, seed=42, season=SEASON, months=24, start=1, loan=4000.0, apr=0.09, term=12,
                  d_sd=150.0, p=P, v=V, d_mu=Q, fixed=F):
    """Cash change since the start, month by month, for n simulated years (shape n x months)."""
    season = check_season(season)
    rng = np.random.default_rng(seed)
    mean_d = np.array([d_mu * season[(start - 1 + m) % 12] for m in range(months)])
    dem = np.maximum(mean_d + d_sd * rng.standard_normal((n, months)), 0.0)
    pay = loan_payment(loan, apr, term) if loan > 0 else 0.0
    inst = np.where(np.arange(months) < term, pay, 0.0)
    return np.cumsum((p - v) * dem - fixed - inst, axis=1)


def shortfall_probability(buffer, **kw):
    """Share of simulated years in which the extra cash needed is more than `buffer` dollars."""
    cs = simulate_cash(**kw)
    return float(np.mean(-cs.min(axis=1) > buffer))

## Step 6: choose the loan (the mini-project engine)
For a loan size and a term, report the instalment, the total interest, the extra cash needed in
the planned (non-random) year and the chance that more than the owner's buffer is needed.

In [ ]:
def evaluate_loan(loan, term, apr=0.09, buffer=1000.0, start=1, season=SEASON, n=5000, seed=42):
    profits = monthly_profits(season, 24, start)
    return {"loan": loan, "term": term, "payment": loan_payment(loan, apr, term) if loan > 0 else 0.0,
            "interest": total_interest(loan, apr, term) if loan > 0 else 0.0,
            "extra": extra_cash_needed(profits, loan, apr, term),
            "risk": shortfall_probability(buffer, n=n, seed=seed, season=season, start=start, loan=loan, apr=apr, term=term)}


def loan_passes(r, buffer=1000.0, max_risk=0.10):
    """Mini-project rule: the planned year needs at most `buffer` of extra cash and the chance of needing more than `buffer` is at most 10 %."""
    return r["extra"] <= buffer and r["risk"] <= max_risk

## See it
Figure 1: the twelve monthly profits against the instalment of a 4000 dollar, 12-month loan.
Figure 2: the running cash total of the seasonal bakery and of a flat bakery with the same
yearly profit.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    prof = monthly_profits()
    print("monthly profits:", np.round(prof[:12], 1), "sum", round(prof[:12].sum(), 1), "flat x 12", 12 * (P - V) * Q - 12 * F)
    for n in (6, 12, 18, 24):
        a = loan_payment(4000, 0.09, n)
        print(f"4000 at 9 %, {n:2d} months: instalment {a:7.2f}  interest {total_interest(4000, 0.09, n):6.1f}  extra cash {extra_cash_needed(prof, 4000, 0.09, n):6.1f}  lowest month {lowest_month(prof, 4000, 0.09, n)}")
    print("extra cash by opening month (4000, 12 months):", {k: round(v_, 1) for k, v_ in extra_cash_by_start(4000, 0.09, 12).items()})
    flat = monthly_profits(scale_season(SEASON, 0.0))
    print("flat year extra cash:", extra_cash_needed(flat, 4000, 0.09, 12))
    for loan in (2000, 3000, 4000, 5000):
        for term in (6, 12, 18, 24):
            r = evaluate_loan(loan, term)
            print(f"loan {loan} term {term:2d}: instalment {r['payment']:6.1f} interest {r['interest']:6.1f} extra {r['extra']:6.1f} risk {100 * r['risk']:5.1f} % {'PASS' if loan_passes(r) else ''}")

    BLUE, ORANGE, MAGENTA = "#0072b2", "#c86a00", "#d81b60"
    pay12 = loan_payment(4000, 0.09, 12)
    fig, ax = plt.subplots(figsize=(6.4, 4.0))
    ax.bar(range(12), prof[:12], color=[BLUE if x >= pay12 else MAGENTA for x in prof[:12]])
    ax.axhline(pay12, color="black", ls="--", lw=1.5)
    ax.text(0.1, pay12 + 25, f"instalment {pay12:.0f} dollars (4000 at 9 %, 12 months)", fontsize=8)
    ax.set_xticks(range(12))
    ax.set_xticklabels(MONTH_NAMES, fontsize=8)
    ax.set_ylabel("monthly profit ($)")
    ax.set_title("Five months do not cover the instalment (magenta)", fontsize=10)
    ax.grid(alpha=0.3, axis="y")
    fig.tight_layout()
    fig.savefig(OUT / "monthly_profit.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.0))
    months = np.arange(0, 25)
    for pr, col, ls, lab in ((prof, BLUE, "-", "seasonal bakery"), (flat, ORANGE, "--", "flat bakery, same yearly profit")):
        cs = np.concatenate([[0.0], np.cumsum(pr - np.where(np.arange(24) < 12, pay12, 0.0))])
        ax2.plot(months, cs, color=col, ls=ls, lw=2, marker="o", ms=3, label=lab)
    ax2.axhline(0, color="black", lw=0.8)
    ax2.set_xlabel("months since opening in January")
    ax2.set_ylabel("cash change since the start ($)")
    ax2.set_title("Same yearly profit, but the seasonal one dips by 365 dollars", fontsize=10)
    ax2.legend(fontsize=8)
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "cash_dip.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `scale_season(SEASON, k)` for k = 0, 0.5, 1 and 1.5 (k = 0 is a flat year, k = 1 the pattern of the lesson) and `extra_cash_needed(monthly_profits(season), 4000, 0.09, 12)` to find the extra cash needed for the 4000 dollar, 12-month loan. Write down the four numbers. Is the extra cash needed proportional to k?
2. **Stretch.** For a 5000 dollar loan at 9 percent, find with `extra_cash_needed` the shortest term (6 to 24 months) that needs at most 200 dollars of extra cash, and its `total_interest`. Then repeat for an interest rate of 0 and of 20 percent and say whether the rate changes the term you pick. Explain why a longer term lowers the extra cash needed although it raises the interest.
3. **Explain.** A friend says: "the bakery earns 5760 dollars a year and the loan instalments are only 4198 dollars a year, so it can never run short of cash". Explain with the running-cash figure what is wrong, and say what the friend would have to know about the months to be right.

## Assessment
- **A. Conceptual.** A seasonal bakery and a flat bakery earn exactly the same profit over a year. Why can only the seasonal one run short of cash when it repays a loan in equal instalments?
- **B. Numerical.** A loan of 3000 dollars at 12 percent a year is repaid over 12 months. Find the monthly instalment, the interest in the first instalment, the balance after it, and the total interest.
- **C. Graph and simulation interpretation.** In the running-cash figure the seasonal and flat curves are 756 dollars apart in March (-365 against +391) but touch again in December. What does the gap in March mean, and why do the curves meet at month 12?
- **D. Design and optimisation challenge.** Design: the owner borrows 4000 dollars at 9 percent and can find at most 200 dollars of extra cash in the planned year. Choose the shortest loan term among 6, 12, 18 and 24 months that works. What does it cost in interest compared with 12 months, and how risky is it when demand is random?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def _raises(fn):
    try:
        fn()
    except ValueError:
        return True
    return False


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    pr = L.monthly_profits()
    flat_month = (L.P - L.V) * L.Q - L.F
    # --- the pattern and the roadmap checkpoint
    checks.append(("the 12 seasonal factors average exactly 1.0 and December is the peak (1.55)",
                   abs(sum(L.SEASON) / 12 - 1.0) < 1e-12 and L.SEASON.index(max(L.SEASON)) == 11))
    total = float(pr[:12].sum())
    checks.append((f"roadmap check: the twelve monthly profits add up to ${total:.0f}, 12 times the flat monthly profit ${flat_month:.0f} (within 1 percent)",
                   abs(total - 12 * flat_month) < 0.01 * 12 * flat_month and abs(total - 5760.0) < 1e-6))
    checks.append(("worked example: monthly profits are 228, 144, 312, 396, 480, 396, 312, 312, 480, 564, 732 and 1404 dollars",
                   np.allclose(pr[:12], [228, 144, 312, 396, 480, 396, 312, 312, 480, 564, 732, 1404])))
    checks.append(("the pattern repeats: month 13 equals month 1 and the profits are a straight line in the factor",
                   abs(pr[12] - pr[0]) < 1e-9 and np.allclose(pr[:12], 2.1 * 800 * np.array(L.SEASON) - 1200)))
    checks.append(("the total over any 12 consecutive months does not depend on the opening month",
                   all(abs(L.monthly_profits(L.SEASON, 12, s).sum() - 5760.0) < 1e-6 for s in range(1, 13))))
    checks.append(("limit: a pattern that does not average 1.0, has the wrong length or a negative factor is rejected",
                   _raises(lambda: L.check_season([1.0] * 11)) and _raises(lambda: L.check_season([1.2] * 12)) and _raises(lambda: L.check_season([-0.5] + [1.0454545454545454] * 11))))
    flat = L.scale_season(L.SEASON, 0.0)
    checks.append(("scale_season: k = 0 is a flat year, k = 1 is unchanged, k = 1.5 still averages 1.0, k = 6 is rejected",
                   all(abs(x - 1.0) < 1e-12 for x in flat) and np.allclose(L.scale_season(L.SEASON, 1.0), L.SEASON)
                   and abs(sum(L.scale_season(L.SEASON, 1.5)) / 12 - 1.0) < 1e-9 and _raises(lambda: L.scale_season(L.SEASON, 6.0))))
    # --- the loan
    a12 = L.loan_payment(4000, 0.09, 12)
    checks.append((f"worked example: 4000 dollars at 9 percent over 12 months is repaid by instalments of ${a12:.2f}", abs(a12 - 349.81) < 0.005))
    sch = L.loan_schedule(4000, 0.09, 12)
    checks.append(("validation: the schedule ends with a balance of 0 and the repaid parts add up to the 4000 borrowed",
                   abs(sch[-1]["balance"]) < 1e-6 and abs(sum(r["repaid"] for r in sch) - 4000.0) < 1e-6))
    checks.append(("the first instalment holds 30.00 dollars of interest (4000 x 0.0075) and the interest falls every month",
                   abs(sch[0]["interest"] - 30.0) < 1e-9 and all(b["interest"] < a["interest"] for a, b in zip(sch, sch[1:]))))
    checks.append((f"total interest is ${L.total_interest(4000, 0.09, 12):.1f} for 12 months and ${L.total_interest(4000, 0.09, 24):.1f} for 24: a longer loan costs more interest but has a smaller instalment",
                   abs(L.total_interest(4000, 0.09, 12) - 197.7) < 0.1 and abs(L.total_interest(4000, 0.09, 24) - 385.7) < 0.1
                   and L.loan_payment(4000, 0.09, 24) < a12))
    checks.append(("limit: with a rate of 0 the instalment is the principal over the term (1000 for 4000 over 4 months)", abs(L.loan_payment(4000, 0.0, 4) - 1000.0) < 1e-12))
    checks.append(("limit: a zero or negative term and a negative rate are rejected", _raises(lambda: L.loan_payment(1000, 0.09, 0)) and _raises(lambda: L.loan_payment(1000, -0.01, 12))))
    # --- the running total
    rt = L.running_total(pr, loan=4000, term=12)
    checks.append((f"month by month: the running total starts at -2000 (the owner's part) and is {rt[0]:.0f}, {rt[1]:.0f}, {rt[2]:.0f} after January, February and March",
                   len(rt) == 24 and abs(rt[0] + 2121.81) < 0.01 and abs(rt[1] + 2327.61) < 0.01 and abs(rt[2] + 2365.42) < 0.01))
    m, low = L.lowest_month(pr, loan=4000, term=12)
    checks.append((f"worked example: the lowest point is in month {m}, ${-low:.1f} below the starting level, so the owner needs ${L.extra_cash_needed(pr, 4000, 0.09, 12):.1f} of extra cash",
                   m == 3 and abs(-low - 365.4) < 0.1 and abs(L.extra_cash_needed(pr, 4000, 0.09, 12) - 365.4) < 0.1))
    checks.append(("a flat bakery with the same yearly profit never dips: the extra cash needed is 0, a seasonal one needs 365.4",
                   L.extra_cash_needed(L.monthly_profits(flat), 4000, 0.09, 12) == 0.0 and abs(L.extra_cash_needed(pr, 4000, 0.09, 12) - 365.4) < 0.1))
    checks.append(("the first year's cash change is the profit total minus 12 instalments: 5760 - 12 x 349.81 = 1562.3",
                   abs(float(np.cumsum(pr - np.where(np.arange(24) < 12, a12, 0.0))[11]) - 1562.33) < 0.01))
    need = [L.extra_cash_needed(pr, 4000, 0.09, t) for t in (6, 12, 18, 24)]
    checks.append((f"a longer term needs less extra cash: {need[0]:.0f}, {need[1]:.0f}, {need[2]:.0f} and {need[3]:.0f} dollars for 6, 12, 18 and 24 months",
                   need[0] > need[1] > need[2] > need[3] == 0.0 and abs(need[2] - 104.8) < 0.1))
    checks.append(("no loan: the running total starts at -6000 and the owner's money is back in month 14; with 4000 over 12 months in month 15",
                   abs(L.running_total(pr, loan=0.0)[0] - (-6000 + pr[0])) < 1e-9 and L.equity_recovered_month(pr, loan=0.0) == 14
                   and L.equity_recovered_month(pr, loan=4000, term=12) == 15))
    by = L.extra_cash_by_start(4000, 0.09, 12)
    checks.append(("the opening month matters: January needs 365.4 dollars, April, May and September to December need none",
                   abs(by[1] - 365.4) < 0.1 and all(by[s] == 0.0 for s in (4, 5, 9, 10, 11, 12))))
    # --- random demand
    cs0 = L.simulate_cash(n=10, d_sd=0.0)
    checks.append(("limit: with no randomness every simulated year equals the planned year",
                   np.allclose(cs0, np.cumsum(pr - np.where(np.arange(24) < 12, a12, 0.0)))))
    mean_sim = L.simulate_cash(n=50000).mean(axis=0)
    plan = np.cumsum(pr - np.where(np.arange(24) < 12, a12, 0.0))
    checks.append((f"validation: the mean of 50000 simulated years follows the plan to within ${np.abs(mean_sim - plan).max():.1f}", np.abs(mean_sim - plan).max() < 12.0))
    checks.append(("seeded run: the same numbers every time", L.shortfall_probability(500) == L.shortfall_probability(500)))
    s_loan = [L.shortfall_probability(1000, loan=4000, term=t) for t in (6, 12, 18, 24)]
    checks.append((f"the chance of needing more than 1000 dollars falls with the term: {', '.join(f'{100 * x:.1f}' for x in s_loan)} percent for 6, 12, 18 and 24 months",
                   all(b < a for a, b in zip(s_loan, s_loan[1:])) and abs(s_loan[2] - 0.087) < 0.01))
    checks.append(("with no loan the chance of needing more than 200 dollars is about 15 percent: demand noise alone causes loss months in winter",
                   abs(L.shortfall_probability(200, loan=0.0) - 0.152) < 0.01))
    # --- mini-project
    grid = {(loan, t): L.evaluate_loan(loan, t) for loan in (1000, 2000, 3000, 4000, 5000, 6000) for t in (6, 12, 18, 24)}
    ok = [k for k, r in grid.items() if L.loan_passes(r)]
    best = max(ok, key=lambda k: (k[0], -grid[k]["interest"]))
    checks.append((f"mini-project: {len(ok)} of the 24 loan choices pass; the one that borrows the most is {best[0]} dollars over {best[1]} months",
                   len(ok) == 12 and best == (5000, 24)))
    r = grid[(5000, 24)]
    checks.append((f"mini-project: 5000 over 24 months has an instalment of ${r['payment']:.1f}, interest ${r['interest']:.1f}, extra cash ${r['extra']:.1f} and a {100 * r['risk']:.1f} percent chance of needing more than the 1000 buffer",
                   abs(r["payment"] - 228.4) < 0.1 and abs(r["interest"] - 482.2) < 0.1 and abs(r["extra"] - 84.8) < 0.1 and abs(r["risk"] - 0.076) < 0.01))
    checks.append(("mini-project: 5000 over 18 months and 4000 over 12 months fail (risk 19 and 33 percent)",
                   not L.loan_passes(grid[(5000, 18)]) and not L.loan_passes(grid[(4000, 12)])))
    checks.append(("edge: no loan means no instalment and no interest", L.evaluate_loan(0.0, 12)["payment"] == 0.0 and L.evaluate_loan(0.0, 12)["interest"] == 0.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
